# H.4.1 -- per-Reserve-Bank ("regional") statement of condition continuous series

Target: `summary="STATEMENT OF CONDITION OF EACH FEDERAL RESERVE BANK"`,
which matches both the Assets sub-table and the Liabilities/Capital
sub-table (one substring search catches both).

Context/caveat (discussed before building this): the 12-bank split is
mostly an accounting/operational artifact, not 12 independent regional
policies -- securities holdings will sit almost entirely on the New York
Fed's books since NY executes open-market operations for the whole System.
The genuinely informative columns here are things like discount-window
borrowing by district and currency/Federal Reserve notes by district, not
securities holdings by district. Worth keeping in mind for whatever this
ends up being used for (noted as a possible angle for a future, separate
project, not this one).

This table needed two fixes to the shared parser (`Helpers/h41_parse_helpers.py`)
that weren't needed for the balance sheet or maturity tables:
- **A spurious third matched table.** The modern format's footnotes are
  published as their own small `<table>` (footnote number -> explanatory
  prose), which sits immediately after the real Liabilities table and
  inherits proximity to the same "(continued)" caption -- causing it to
  falsely match the heading-text search too, with its footnote prose
  becoming a nonsense "column." Fixed by detecting and excluding tables
  whose first column is mostly bare footnote markers ("1.", "2.", ...).
- **Genuine duplicate rows across the Assets/Liabilities tables.** Several
  line items (confirmed: Gold certificate account, SDR certificate
  account) are reported twice -- once with a real per-Bank breakdown in
  one table, once as a Total-only memo cross-reference in the other,
  sometimes under a slightly different label spelling. Concatenating both
  verbatim silently doubled the Total column on `groupby().sum()` while
  leaving individual Banks correct (since the memo copy's own Bank columns
  are blank). Fixed two ways: `extract_tidy` now dedupes identical
  (date, row_label, col_label) triples before concatenating multiple
  matched tables; the crosswalk-application step here additionally dedupes
  exact (date, line_item, bank, value) matches post-mapping, since the two
  spellings only collide after crosswalk mapping.
- **A stray trailing semicolon** in some 1997-1999 release cells (e.g.
  `"34,713 ;"`) made `clean_value` correctly flag them as unparseable --
  but the resulting NaN then silently summed as 0 in the wide pivot,
  which is how `TOTAL_LIABILITIES` and `OTHER_LIABILITIES_AND_ACCRUED_DIVIDENDS`
  picked up real (not rounding-level) reconciliation failures confined to
  that narrow window. Confirmed widespread enough (191 of 300 unparseable
  cells) to be a systematic source quirk, not a typo -- `clean_value` now
  strips it. Also added parenthesized-negative accounting notation
  (`"(19)"` -> `-19`) while in there.

Known, confirmed-legitimate (not bugs) reconciliation gaps, left as-is:
a handful of pre-2002-only aggregates (`COLLATERAL_TOTAL_PRE2002`,
`UST_AND_AGENCY_SECURITIES_PRE2002`, `OTHER_ELIGIBLE_ASSETS_PRE2002`) are
genuinely only ever reported at the System (Total) level in the source,
with every individual Bank column reported as a true zero -- not a parsing
gap, confirmed by inspecting the raw grid directly.

In [1]:
import os, sys, glob
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
from h41_parse_helpers import (
    extract_tidy_many, add_clean_value, label_inventory, unmapped_labels,
    load_or_init_crosswalk, apply_crosswalk, check_components_sum_to_total,
    coverage_report, save_wide_excel,
)
import pandas as pd

BASE = os.path.dirname(os.getcwd())
HTML_DIR = os.path.join(BASE, "Data Input", "Raw Input", "h41_html")
CROSSWALK_DIR = os.path.join(BASE, "Data Input", "Raw Input", "crosswalks")
os.makedirs(CROSSWALK_DIR, exist_ok=True)
ROW_CROSSWALK_PATH = os.path.join(CROSSWALK_DIR, "regional_row_crosswalk.xlsx")  # line items
COL_CROSSWALK_PATH = os.path.join(CROSSWALK_DIR, "regional_col_crosswalk.xlsx")  # bank names

SUMMARY_TARGET = "STATEMENT OF CONDITION OF EACH FEDERAL RESERVE BANK"

all_files = sorted(glob.glob(os.path.join(HTML_DIR, "h41_*.html")))
print(f"{len(all_files)} release files found")

1544 release files found


## Step 1 -- extract every release to long/tidy form

In [2]:
tidy = extract_tidy_many(all_files, SUMMARY_TARGET)
tidy = add_clean_value(tidy)
print(f"{len(tidy):,} tidy rows across {tidy['date'].nunique()} releases")
print(f"parse_failed rate: {tidy['parse_failed'].mean():.2%}")
tidy.head(15)

787,111 tidy rows across 1543 releases
parse_failed rate: 0.01%


,date,row_label_raw,col_label_raw,value_raw,source,value,parse_failed
0,1997-03-05 00:00:00,Gold certificate account,Total,"11,051",h41_19970306.html,11051.0,False
1,1997-03-05 00:00:00,Gold certificate account,Boston,661,h41_19970306.html,661.0,False
2,1997-03-05 00:00:00,Gold certificate account,New York,"4,052",h41_19970306.html,4052.0,False
3,1997-03-05 00:00:00,Gold certificate account,Philadelphia,423,h41_19970306.html,423.0,False
4,1997-03-05 00:00:00,Gold certificate account,Cleveland,624,h41_19970306.html,624.0,False
5,1997-03-05 00:00:00,Gold certificate account,Richmond,919,h41_19970306.html,919.0,False
6,1997-03-05 00:00:00,Gold certificate account,Atlanta,769,h41_19970306.html,769.0,False
7,1997-03-05 00:00:00,Gold certificate account,Chicago,"1,140",h41_19970306.html,1140.0,False
8,1997-03-05 00:00:00,Gold certificate account,St. Louis,474,h41_19970306.html,474.0,False
9,1997-03-05 00:00:00,Gold certificate account,Minneapolis,168,h41_19970306.html,168.0,False


## Step 2 -- label inventory and crosswalk gap detection

Column labels converge to 13 values: the 12 Reserve Banks plus "Total" --
watch for bank-name variants (abbreviation typos genuinely present in a
couple of 1997-1999 source releases, and "Kansas City"/"San Francisco"
wrapping across two header rows in the modern format). Row labels are the
line items; old-era (pre-Dec-2002) terminology gets mapped to the same
standardized category as its modern renamed equivalent wherever the two
are confirmed to mean the same thing, and kept as a separate pre-2002-only
category where they're confirmed NOT to (different values on overlapping
dates -- checked directly, not assumed).

In [3]:
row_inventory = label_inventory(tidy, "row_label_raw")
col_inventory = label_inventory(tidy, "col_label_raw")

row_crosswalk = load_or_init_crosswalk(ROW_CROSSWALK_PATH)
col_crosswalk = load_or_init_crosswalk(COL_CROSSWALK_PATH)

row_unmapped = unmapped_labels(row_inventory, row_crosswalk, "row_label_raw")
col_unmapped = unmapped_labels(col_inventory, col_crosswalk, "col_label_raw")

print(f"{len(row_unmapped)} / {len(row_inventory)} row (line item) labels unmapped")
print(f"{len(col_unmapped)} / {len(col_inventory)} column (bank name) labels unmapped -- should settle to ~13")

row_unmapped.rename(columns={"row_label_raw": "raw"}).assign(standardized="").to_excel(
    os.path.join(CROSSWALK_DIR, "regional_row_UNMAPPED_review.xlsx"), index=False)
col_unmapped.rename(columns={"col_label_raw": "raw"}).assign(standardized="").to_excel(
    os.path.join(CROSSWALK_DIR, "regional_col_UNMAPPED_review.xlsx"), index=False)
col_unmapped

0 / 149 row (line item) labels unmapped
0 / 37 column (bank name) labels unmapped -- should settle to ~13


,col_label_raw,first_seen,last_seen,n_obs


## Step 3 -- apply crosswalk (once populated)

A second, narrower dedup pass here: two different raw labels can map to
the same standardized line item and both report the exact same value for
a given (date, bank) -- the memo-cross-reference pattern confirmed for
SDR_CERTIFICATE_ACCOUNT (two different spellings, one on the Assets side,
one a Total-only memo on the Liabilities side, same Total figure in both
but under slightly different wording so the raw-string dedup in
`extract_tidy` can't catch it). Genuinely distinct additive components
essentially never coincide in value by chance, so dedup on the full
(date, line_item, bank, value) tuple before summing.

In [4]:
tidy_mapped = apply_crosswalk(tidy, row_crosswalk, "row_label_raw", out_col="line_item")
tidy_mapped = apply_crosswalk(tidy_mapped, col_crosswalk, "col_label_raw", out_col="bank")

fully_mapped = tidy_mapped.dropna(subset=["line_item", "bank"]).copy()
before = len(fully_mapped)
fully_mapped = fully_mapped.drop_duplicates(subset=["date", "line_item", "bank", "value"])
print(f"{len(fully_mapped):,} / {len(tidy_mapped):,} rows fully mapped on both axes "
      f"({len(fully_mapped)/len(tidy_mapped):.2%}), dropped {before - len(fully_mapped):,} "
      f"exact (date, line_item, bank, value) duplicates")

742,257 / 787,111 rows fully mapped on both axes (94.30%), dropped 16,176 exact (date, line_item, bank, value) duplicates


## Step 4 -- pivot to wide, once the crosswalk covers enough of the data

In [5]:
wide = (
    fully_mapped
    .groupby(["date", "line_item", "bank"])["value"].sum()
    .unstack("bank")
    .sort_index()
)
print(f"wide shape: {wide.shape}, {wide.index.get_level_values('line_item').nunique()} line items")
wide.head()

wide shape: (55187, 13), 67 line items


bank                               ATLANTA  BOSTON  CHICAGO  CLEVELAND  \
date       line_item                                                     
1997-03-05 ACCEPTANCES                 0.0     0.0      0.0        0.0   
           BANK_PREMISES              80.0    95.0    109.0      111.0   
           CAPITAL_ACCOUNTS_TOTAL      0.0     0.0      0.0        0.0   
           CAPITAL_PAID_IN           463.0   178.0    547.0      283.0   
           COIN                       94.0    21.0     81.0       35.0   

bank                               DALLAS  KANSAS_CITY  MINNEAPOLIS  NEW_YORK  \
date       line_item                                                            
1997-03-05 ACCEPTANCES                0.0          0.0          0.0       0.0   
           BANK_PREMISES            154.0         56.0        119.0     153.0   
           CAPITAL_ACCOUNTS_TOTAL     0.0          0.0          0.0       0.0   
           CAPITAL_PAID_IN          258.0        179.0        110.0    1078.0   
           COIN                      59.0         58.0         26.0      35.0   

bank                               PHILADELPHIA  RICHMOND  SAN_FRANCISCO  \
date       line_item                                                       
1997-03-05 ACCEPTANCES                      0.0       0.0            0.0   
           BANK_PREMISES                   50.0     127.0          160.0   
           CAPITAL_ACCOUNTS_TOTAL           0.0       0.0            0.0   
           CAPITAL_PAID_IN                285.0     313.0          923.0   
           COIN                            61.0     123.0           90.0   

bank                               ST_LOUIS   TOTAL  
date       line_item                                 
1997-03-05 ACCEPTANCES                  0.0     0.0  
           BANK_PREMISES               30.0  1245.0  
           CAPITAL_ACCOUNTS_TOTAL       0.0     0.0  
           CAPITAL_PAID_IN            115.0  4732.0  
           COIN                        33.0   716.0

## Step 5 -- reconciliation: 12 banks should sum to "Total"

This is reported as an independent column in the same table, so it's a free
cross-check rather than something needing a separate source.

In [6]:
BANKS = ["BOSTON", "NEW_YORK", "PHILADELPHIA", "CLEVELAND", "RICHMOND", "ATLANTA",
         "CHICAGO", "ST_LOUIS", "MINNEAPOLIS", "KANSAS_CITY", "DALLAS", "SAN_FRANCISCO"]

recon_results = {}
for item in wide.index.get_level_values("line_item").unique():
    sub = wide.xs(item, level="line_item")
    have_banks = [b for b in BANKS if b in sub.columns]
    if "TOTAL" not in sub.columns or len(have_banks) < 8:
        continue
    bad_weeks = check_components_sum_to_total(sub, component_cols=have_banks, total_col="TOTAL", tolerance=5.0)
    recon_results[item] = (len(sub), len(bad_weeks))

recon_df = pd.DataFrame(
    [(item, n, bad, bad / n) for item, (n, bad) in recon_results.items()],
    columns=["line_item", "n_weeks", "bad_weeks", "pct_bad"],
).sort_values("pct_bad", ascending=False)
print(recon_df.to_string(index=False))
print(f"\n{(recon_df.bad_weeks == 0).sum()} / {len(recon_df)} line items reconcile with zero bad weeks")

                                     line_item  n_weeks  bad_weeks  pct_bad
             UST_AND_AGENCY_SECURITIES_PRE2002      300        299 0.996667
                      COLLATERAL_TOTAL_PRE2002      298        297 0.996644
                 OTHER_ELIGIBLE_ASSETS_PRE2002      300         50 0.166667
                  UST_SECURITIES_TOTAL_PRE2002      300          1 0.003333
            LOANS_AND_SECURITIES_TOTAL_PRE2002      300          1 0.003333
                                  TOTAL_ASSETS     1541          5 0.003245
                     FOREIGN_OFFICIAL_DEPOSITS     1541          3 0.001947
                      TREASURY_GENERAL_ACCOUNT     1541          1 0.000649
                        CAPITAL_ACCOUNTS_TOTAL     1541          0 0.000000
                                   ACCEPTANCES      300          0 0.000000
                                 BANK_PREMISES     1200          0 0.000000
             FEDERAL_AGENCY_SECURITIES_PRE2002      617          0 0.000000
            

## Step 6 -- save to Data Output

Tidy format: `date`, `line_item`, `bank`, `value`. `TOTAL` rows are kept
in (the table's own reported System-wide figure, used above for
reconciliation, not re-derived).

In [7]:
out_cols = ["date", "line_item", "bank", "value"]
out = fully_mapped[out_cols].sort_values(["line_item", "bank", "date"]).reset_index(drop=True)

out_path = os.path.join(BASE, "Data Output", "H41_regional_clean_timeseries.xlsx")
out.to_excel(out_path, index=False)
print(f"Saved {len(out):,} rows to: {out_path}")

Saved 742,257 rows to: C:\Users\User\Desktop\DISSERTATION PAPERS\PAPER 1 - STRUCTURAL BREAKS\Data Output\H41_regional_clean_timeseries.xlsx


## Step 7 -- pivot to the paper's wide deliverable shape

Dates in columns, `(line_item, bank)` in rows -- matching JHU's own spreadsheet structure. `TOTAL` is kept in as its own bank column (the table's reported System-wide figure).

In [8]:
wide_out_path = os.path.join(BASE, "Data Output", "H41_regional_1997_2026_levels_wide.xlsx")
wide_final = save_wide_excel(fully_mapped, index_cols=["line_item", "bank"], out_path=wide_out_path)

Saved 871 rows x 1,541 date columns to: C:\Users\User\Desktop\DISSERTATION PAPERS\PAPER 1 - STRUCTURAL BREAKS\Data Output\H41_regional_1997_2026_levels_wide.xlsx
